# Pendahuluan

Di dalam material ini, kamu akan mempelajari Data Pima Indians Diabetes dan melakukan pemodelan. Jalankan Capstone 1 terlebih dahulu karena notebook ini memakai file `data.csv` hasil pembersihan dari Capstone 1.

In [ ]:
# Tahapan sebelumnya
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

# Memuat dataset yang sudah rapi ditahapan sebelumnya
data = "data.csv"
df = pd.read_csv(data, index_col=0)
df['Outcome'] = df['Outcome'].replace({'Non-Diabetes': 0, 'Diabetes': 1}).astype(int)

# Variabel terpilih
df = df[['Glucose', 'BMI', 'Age', 'Pregnancies', 'DiabetesPedigreeFunction', 'BloodPressure', 'Outcome']]

In [ ]:
# Liat informasi dari dataset yang kita miliki
print(df.head())
print(df.columns)

**Memisahkan variabel dependen dan independen**

Variabel dependen adalah kolom 'Outcome', sedangkan kolom lainnya merupakan variabel independen. Variabel X dibuat menggunakan `.isin()` untuk semua kolom kecuali Outcome, dan variabel y hanya berisi kolom Outcome.

In [ ]:
# Define X and y
X = df.loc[:, ~df.columns.isin(['Outcome'])]
y = df['Outcome']

In [ ]:
# Melihat kolom yang termasuk variable independen
X.head()

In [ ]:
# Melihat variabel dependen
y.head()

**Spliting data**

Dataset dipisah menjadi training dan testing dengan perbandingan 80:20 dan random state 42. Parameter `stratify=y` dipakai agar proporsi Diabetes dan Non-Diabetes (sekitar 2:1) sama di data training dan testing.

In [ ]:
# Mendefinisikan pemisahan data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Menampilkan hasil pemisahan berdasarkan kolom dan baris
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

**Scaling data**

Scaler di-*fit* **hanya pada data training**, lalu dipakai untuk mentransformasi data testing. Jika scaler di-fit pada seluruh data sebelum split, informasi rata-rata dan simpangan baku data testing ikut "bocor" ke proses training (*data leakage*).

In [ ]:
# Mendefinisikan standard scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

**Pemodelan dengan Diabetes Dataset**

Kita akan melakukan pemodelan untuk 4 model berbeda: logistic regression, decision tree, random forest, dan neural network (Multi-layer Perceptron).

In [ ]:
# Menyiapkan libraries
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
clf = LogisticRegression()
# train the classifier
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
# printing the test accuracy
print("The test accuracy score of Logistic Regression Classifier is ", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
clf = DecisionTreeClassifier(random_state=42)
# train the classifier
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
# printing the test accuracy
print("The test accuracy score of Decision Tree Classifier is ", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
clf = RandomForestClassifier(random_state=42)
# train the classifier
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
# printing the test accuracy
print("The test accuracy score of Random Forest Classifier is ", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
clf = MLPClassifier(random_state=42, max_iter=1000)
# train the classifier
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
# printing the test accuracy
print("The test accuracy score of MLP Classifier is ", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

**Membandingkan hasil di atas (model default, tanpa tuning)**

Berikut akurasi dan recall kelas Diabetes (kelas 1) pada data testing (142 pasien, 47 di antaranya Diabetes):
- Logistic Regression: akurasi 77%, recall 53%.
- Decision Tree: akurasi terendah 69%, recall 49%.
- Random Forest: akurasi 77%, recall 51%.
- MLP: akurasi 76%, recall tertinggi 60%.

Akurasi ketiga model terbaik hampir sama, tetapi recall kelas Diabetes masih rendah: sekitar separuh penderita diabetes tidak terdeteksi. Ini konsekuensi data yang tidak seimbang, karena model cenderung memihak kelas mayoritas (Non-Diabetes). Decision Tree tanpa batasan kedalaman juga cenderung overfitting pada data training, sehingga performanya paling rendah di data testing.

**Pojok Catatan**

Recall adalah kemampuan model untuk menemukan semua kasus positif. Untuk skrining penyakit, recall pada kelas positif lebih penting daripada presisi: *false negative* berarti penderita diabetes dinyatakan sehat dan tidak mendapat penanganan, sedangkan *false positive* hanya berujung pada pemeriksaan lanjutan (misalnya tes HbA1c) yang akan mengoreksi kesalahan itu.

Karena data tidak seimbang (sekitar 2:1), grid search di bawah juga mencoba `class_weight='balanced'` pada Logistic Regression, Decision Tree, dan Random Forest. Opsi ini memberi bobot lebih besar pada kelas minoritas (Diabetes) sehingga recall kelas tersebut cenderung naik.

**Proses Hyperparameter Tuning**

Tuning memakai GridSearchCV dengan 5-fold cross validation dan metrik ROC-AUC. Pemilihan parameter hanya memakai data training; data testing baru dipakai untuk evaluasi akhir.

In [ ]:
# LogisticRegression with hyperparameter
clf = LogisticRegression()

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver' : ['lbfgs', 'liblinear'],
    'class_weight': [None, 'balanced'],
    'max_iter': [1000]
}

gs1 = GridSearchCV(
        estimator=clf,
        param_grid = param_grid,
        cv=5,
        n_jobs=-1,
        scoring='roc_auc'
    )

fit_clf_lg = gs1.fit(X_train, y_train)

print(fit_clf_lg.best_params_)
print(fit_clf_lg.best_score_)

In [ ]:
y_pred = fit_clf_lg.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# RandomForestClassifier with hyperparameter
clf = RandomForestClassifier(random_state=42)

param_grid = {'n_estimators': [100, 200, 300],
               'max_depth': [None, 4, 6, 8],
               'min_samples_leaf': [1, 5, 10],
               'class_weight': [None, 'balanced']}

gs1 = GridSearchCV(
        estimator=clf,
        param_grid = param_grid,
        cv=5,
        n_jobs=-1,
        scoring='roc_auc'
    )

fit_clf_rf = gs1.fit(X_train, y_train)

print(fit_clf_rf.best_params_)
print(fit_clf_rf.best_score_)

In [ ]:
y_pred = fit_clf_rf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# DecisionTreeClassifier with hyperparameter
clf = DecisionTreeClassifier(random_state=42)

param_grid = {'min_samples_leaf': [1, 5, 10, 20],
               'max_depth': [3, 4, 5, 7, None],
               'criterion': ['gini', 'entropy'],
               'class_weight': [None, 'balanced']}

gs1 = GridSearchCV(
        estimator=clf,
        param_grid = param_grid,
        cv=5,
        n_jobs=-1,
        scoring='roc_auc'
    )

fit_clf_dt = gs1.fit(X_train, y_train)

print(fit_clf_dt.best_params_)
print(fit_clf_dt.best_score_)

In [ ]:
y_pred = fit_clf_dt.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# MLPClassifier with hyperparameter
clf = MLPClassifier(random_state=42, max_iter=1000)

param_grid1 = {'hidden_layer_sizes': [(10,), (50,), (100,), (50, 25)],
               'activation': ['relu', 'tanh'],
               'solver': ['adam', 'sgd'],
               'alpha': [0.0001, 0.01]}

gs1 = GridSearchCV(
        estimator=clf,
        param_grid = param_grid1,
        cv=5,
        n_jobs=-1,
        scoring='roc_auc'
    )

fit_clf_mlp = gs1.fit(X_train, y_train)

print(fit_clf_mlp.best_params_)
print(fit_clf_mlp.best_score_)

In [ ]:
y_pred = fit_clf_mlp.predict(X_test)
print(classification_report(y_test, y_pred))

**Hasil hyperparameter tuning**

- **Logistic Regression**: parameter terbaik C = 1 tanpa class_weight, yang sama dengan model default. Hasil di data testing tidak berubah (akurasi 77%, recall 53%). Model linear sederhana ini ternyata sudah optimal untuk grid yang dicoba.
- **Random Forest**: parameter terbaik memakai `class_weight='balanced'`, max_depth = 4, dan min_samples_leaf = 5. Akurasi turun dari 77% menjadi 73%, tetapi **recall kelas Diabetes naik dari 51% menjadi 68%**. Pembobotan kelas membuat model lebih berani memprediksi Diabetes: lebih banyak penderita terdeteksi, dengan konsekuensi lebih banyak pasien sehat yang salah diprediksi. Untuk tujuan skrining, pertukaran ini menguntungkan.
- **Decision Tree**: pembatasan max_depth = 5 dan min_samples_leaf = 20 menaikkan akurasi dari 69% menjadi 73% karena pohon menjadi lebih sederhana dan tidak terlalu menghafal data training. Recall justru turun ke 43%, sehingga model ini tetap yang terlemah.
- **MLP**: akurasi naik tipis dari 76% menjadi 77%, tetapi recall turun dari 60% menjadi 49%.

Catatan: perubahan akurasi pada data testing tidak bisa dipakai untuk menyimpulkan bahwa model "lebih general" atau "tidak overfitting". Data testing hanya berisi 142 pasien, sehingga selisih satu atau dua persen setara dengan selisih dua sampai tiga pasien saja. Ukuran yang lebih stabil untuk membandingkan model adalah ROC-AUC hasil 5-fold cross validation (`best_score_`).

**ROC Analysis**

Kurva ROC menggambarkan performa *binary classifier* pada berbagai nilai threshold, dengan TPR (True Positive Rate) di sumbu y dan FPR (False Positive Rate) di sumbu x.

In [ ]:
# Predict the probabilities for the positive class
y_pred_logreg = fit_clf_lg.predict_proba(X_test)[:, 1]
y_pred_rf = fit_clf_rf.predict_proba(X_test)[:, 1]
y_pred_dt = fit_clf_dt.predict_proba(X_test)[:, 1]
y_pred_mlp = fit_clf_mlp.predict_proba(X_test)[:, 1]

# Calculate the AUC-ROC scores
auc_logreg = roc_auc_score(y_test, y_pred_logreg)
auc_rf = roc_auc_score(y_test, y_pred_rf)
auc_dt = roc_auc_score(y_test, y_pred_dt)
auc_mlp = roc_auc_score(y_test, y_pred_mlp)

print(f"AUC-ROC for Logistic Regression: {auc_logreg}")
print(f"AUC-ROC for Random Forest: {auc_rf}")
print(f"AUC-ROC for Decision Tree: {auc_dt}")
print(f"AUC-ROC for MLP: {auc_mlp}")

ROC adalah kurva probabilitas dan AUC mengukur seberapa baik model memisahkan kedua kelas. AUC mendekati 1 berarti pemisahan sangat baik, AUC 0,5 berarti model tidak lebih baik dari tebakan acak. AUC sebesar 0,8 dapat dibaca sebagai: jika diambil satu pasien Diabetes dan satu pasien Non-Diabetes secara acak, peluang model memberi skor risiko lebih tinggi kepada pasien Diabetes adalah 80%.

In [ ]:
#Membuat ROC Analisis
def plot_roc_curves(y_test, y_pred_logreg, y_pred_rf, y_pred_dt, y_pred_mlp):
    plt.figure(figsize=(8, 6))

    # Calculate ROC curves for each model
    fpr_logreg, tpr_logreg, _ = roc_curve(y_test, y_pred_logreg)
    fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_rf)
    fpr_dt, tpr_dt, _ = roc_curve(y_test, y_pred_dt)
    fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_pred_mlp)

    # Plot ROC curves
    plt.plot(fpr_logreg, tpr_logreg, label=f'Logistic Regression (AUC = {auc_logreg:.2f})')
    plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.2f})')
    plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {auc_dt:.2f})')
    plt.plot(fpr_mlp, tpr_mlp, label=f'MLP (AUC = {auc_mlp:.2f})')

    # Plot random classifier
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.50)')

    # Format the plot
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Diabetes Prediction Models')
    plt.legend()
    plt.show()

plot_roc_curves(y_test, y_pred_logreg, y_pred_rf, y_pred_dt, y_pred_mlp)

In [ ]:
def find_rates_for_thresholds(y_test, y_pred, thresholds):
    fpr_list = []
    tpr_list = []
    for threshold in thresholds:
        y_pred_binary = (y_pred > threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_binary, labels=[0, 1]).ravel()
        fpr = fp / (fp + tn)
        tpr = tp / (tp + fn)
        fpr_list.append(fpr)
        tpr_list.append(tpr)
    return fpr_list, tpr_list

thresholds = np.arange(0, 1.1, 0.1)

fpr_logreg, tpr_logreg = find_rates_for_thresholds(y_test, y_pred_logreg, thresholds)
fpr_rf, tpr_rf = find_rates_for_thresholds(y_test, y_pred_rf, thresholds)
fpr_dt, tpr_dt = find_rates_for_thresholds(y_test, y_pred_dt, thresholds)
fpr_mlp, tpr_mlp = find_rates_for_thresholds(y_test, y_pred_mlp, thresholds)

# Summary DataFrame
summary_df = pd.DataFrame({
    'Threshold': thresholds.round(1),
    'FPR_LogReg': fpr_logreg, 'TPR_LogReg': tpr_logreg,
    'FPR_RF': fpr_rf, 'TPR_RF': tpr_rf,
    'FPR_DT': fpr_dt, 'TPR_DT': tpr_dt,
    'FPR_MLP': fpr_mlp, 'TPR_MLP': tpr_mlp
})

print(summary_df.round(3))

In [ ]:
def find_best_threshold(y_test, y_pred):
    # based on Youden's Index (J = TPR - FPR)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    optimal_idx = np.argmax(tpr - fpr)
    return thresholds[optimal_idx]

best_threshold_logreg = find_best_threshold(y_test, y_pred_logreg)
best_threshold_rf = find_best_threshold(y_test, y_pred_rf)
best_threshold_dt = find_best_threshold(y_test, y_pred_dt)
best_threshold_mlp = find_best_threshold(y_test, y_pred_mlp)

print(f"Best threshold for Logistic Regression: {best_threshold_logreg}")
print(f"Best threshold for Random Forest: {best_threshold_rf}")
print(f"Best threshold for Decision Tree: {best_threshold_dt}")
print(f"Best threshold for MLP: {best_threshold_mlp}")

In [ ]:
# Ringkasan performa akhir semua model tuned pada data testing (threshold default 0,5)
from sklearn.metrics import f1_score
models = {'Logistic Regression': (fit_clf_lg, y_pred_logreg),
          'Random Forest': (fit_clf_rf, y_pred_rf),
          'Decision Tree': (fit_clf_dt, y_pred_dt),
          'MLP': (fit_clf_mlp, y_pred_mlp)}
rows = []
for name, (m, proba) in models.items():
    pred = m.predict(X_test)
    rows.append({'Model': name,
                 'CV AUC (train)': m.best_score_,
                 'Test AUC': roc_auc_score(y_test, proba),
                 'Accuracy': accuracy_score(y_test, pred),
                 'Precision': precision_score(y_test, pred),
                 'Recall': recall_score(y_test, pred),
                 'F1': f1_score(y_test, pred)})
result_df = pd.DataFrame(rows).set_index('Model').round(3)
result_df

**Ingat**

1. Jika menginginkan sensitivitas tinggi (sebanyak mungkin penderita terdeteksi), gunakan threshold yang lebih rendah, dengan konsekuensi False Positive Rate ikut naik.
2. Jika ingin mengurangi pasien sehat yang salah diprediksi diabetes, gunakan threshold yang lebih tinggi, dengan konsekuensi sensitivitas turun.

**Kesimpulan**

Berdasarkan ROC-AUC cross validation pada data training, Random Forest memiliki skor tertinggi (0,849), diikuti Logistic Regression dan MLP (keduanya 0,845), lalu Decision Tree (0,817). Pada data testing, AUC Logistic Regression (0,834), MLP (0,839), dan Random Forest (0,823) hampir sama; selisih sekecil ini tidak cukup untuk menyatakan satu model unggul secara meyakinkan.

**Random Forest dengan `class_weight='balanced'` dipilih sebagai model akhir** karena dua alasan: (1) skor CV AUC-nya tertinggi, dan pemilihan berdasarkan CV lebih adil daripada berdasarkan data testing; (2) recall kelas Diabetes-nya paling tinggi (68%) pada threshold default, sesuai tujuan skrining yang mengutamakan sedikitnya penderita yang terlewat. Konsekuensinya adalah presisi yang lebih rendah (57%): dari setiap 100 pasien yang diprediksi Diabetes, sekitar 43 sebenarnya tidak menderita diabetes dan perlu dikonfirmasi dengan pemeriksaan lanjutan.

Threshold optimal Random Forest menurut Youden's Index adalah sekitar 0,40. Pada tabel threshold, menurunkan threshold Random Forest dari 0,5 ke 0,4 menaikkan TPR dari 0,68 menjadi 0,85 dengan FPR naik dari 0,25 menjadi 0,35. Dashboard di Capstone 3 menampilkan probabilitas prediksi agar tenaga kesehatan bisa menilai sendiri pasien yang berada dekat ambang batas.

In [ ]:
import pickle
from sklearn.pipeline import make_pipeline

# Model terbaik dipilih berdasarkan ROC-AUC cross validation pada data training
best_name = result_df['CV AUC (train)'].idxmax()
best_model = models[best_name][0].best_estimator_
print("Model yang disimpan:", best_name)

# Menyimpan scaler + model terbaik dalam satu pipeline agar dashboard bisa menerima input mentah
pklname = "generate_diabetes.pkl"

with open(pklname, 'wb') as file:
    pickle.dump(make_pipeline(scaler, best_model), file)

**Menyimpan ringkasan evaluasi untuk dashboard**

Dashboard di Capstone 3 menampilkan perbandingan model, kurva ROC, confusion matrix, dan tingkat kepentingan fitur. Agar dashboard tidak perlu melatih ulang keempat model setiap kali dibuka, hasil evaluasi disimpan ke file `model_report.json`.

In [ ]:
import json

best_proba = models[best_name][1]
fitted_best = models[best_name][0].best_estimator_
if hasattr(fitted_best, 'feature_importances_'):
    importance = dict(zip(X.columns, fitted_best.feature_importances_.round(4).tolist()))
    importance_type = 'feature_importances_'
elif hasattr(fitted_best, 'coef_'):
    importance = dict(zip(X.columns, fitted_best.coef_[0].round(4).tolist()))
    importance_type = 'coef_'
else:
    importance, importance_type = {}, None

best_thr = {'Logistic Regression': best_threshold_logreg, 'Random Forest': best_threshold_rf,
            'Decision Tree': best_threshold_dt, 'MLP': best_threshold_mlp}

report = {
    'best_model': best_name,
    'best_params': models[best_name][0].best_params_,
    'n_train': int(len(y_train)), 'n_test': int(len(y_test)),
    'metrics': result_df.reset_index().to_dict(orient='records'),
    'roc': {name: dict(zip(['fpr', 'tpr'], [v.round(4).tolist() for v in roc_curve(y_test, proba)[:2]]))
            for name, (m, proba) in models.items()},
    'youden_threshold': {k: float(min(v, 1.0)) for k, v in best_thr.items()},
    'confusion_matrix': confusion_matrix(y_test, (best_proba >= 0.5).astype(int)).tolist(),
    'feature_importance': importance, 'importance_type': importance_type,
}
with open('model_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print("model_report.json tersimpan. Model terbaik:", best_name)